# AWS Marketplace Product Usage Demonstration - Model Packages


---

This notebook's CI test result for us-west-2 is as follows. CI test results in other regions can be found at the end of the notebook. 

![This us-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-2/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

---


## Using Model Package ARN with Amazon SageMaker APIs

This sample notebook demonstrates two new functionalities added to Amazon SageMaker:
1. Using a Model Package ARN for inference via Batch Transform jobs / Live Endpoints
2. Using a Marketplace Model Package ARN - we will use [Scikit Decision Trees - Pretrained Model](https://aws.amazon.com/marketplace/pp/prodview-7qop4x5ahrdhe?qid=1543169069960&sr=0-2&ref_=srh_res_product_title)


## Overall flow diagram
<img src="images/ModelPackageE2EFlow.jpg">

## Compatibility
This notebook is compatible only with [Scikit Decision Trees - Pretrained Model](https://aws.amazon.com/marketplace/pp/prodview-7qop4x5ahrdhe?qid=1543169069960&sr=0-2&ref_=srh_res_product_title) sample model that is published to AWS Marketplace

> **SageMaker Python SDK v3 note:** This notebook has been migrated to the SageMaker Python SDK v3. The v2 high-level `sagemaker.ModelPackage` (with `.deploy()` / `.transformer()` helpers) and `sagemaker.predictor.Predictor` have been replaced by the `sagemaker-core` resource classes (`Model`, `EndpointConfig`, `Endpoint`, `TransformJob`). A `sagemaker-core` `Model` is created whose primary container references the Marketplace model-package ARN, and that same model is reused for both the batch transform job and the live endpoint.

## Set up the environment

In [1]:
# v3: `import sagemaker` / `from sagemaker import get_execution_role` ->
# sagemaker-core session helpers. CSVSerializer / Predictor are no longer
# needed: real-time inference is done via `Endpoint.invoke(...)` with csv bytes.
from sagemaker.core.helper.session_helper import Session, get_execution_role

role = "arn:aws:iam::729646638167:role/SageMakerRole"  # [papermill-run] explicit role (local non-SageMaker env)

# S3 prefixes
common_prefix = "DEMO-scikit-byo-iris"
batch_inference_input_prefix = common_prefix + "/batch-inference-input-data"

/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_url" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_data_source" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/pydantic/_internal/_fields.py:128: UserWarning: Field "model_description" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/

### Create the session

The session remembers our connection parameters to Amazon SageMaker. We'll use it to perform all of our Amazon SageMaker operations.

In [2]:
# v3: `sagemaker.Session()` -> `sagemaker.core.helper.session_helper.Session()`.
import boto3  # [papermill-run] region injection
sagemaker_session = Session(boto_session=boto3.Session(region_name="us-west-1"))
region = "us-west-1"  # [papermill-run] pinned region

[07/16/26 12:57:17] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=4179103;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=4179104;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


## Create Model

Now we use the above Model Package to create a model

In [3]:
from src.scikit_product_arns import ScikitArnProvider

modelpackage_arn = ScikitArnProvider.get_model_package_arn(region)
print("Using model package arn " + modelpackage_arn)

Using model package arn arn:aws:sagemaker:us-west-1:382657785993:model-package/scikit-iris-detector-154230595-8f00905c1f927a512b73ea29dd09ae30


In [4]:
# v3: the v2 high-level `ModelPackage(model_package_arn=...)` deployable model +
# its `.deploy()` / `.transformer()` helpers are replaced by sagemaker-core
# resources. We create a core `Model` whose primary container references the
# Marketplace model-package ARN, then reuse it for both the live endpoint and
# the batch transform job below.
from sagemaker.core.resources import Model, EndpointConfig, Endpoint, TransformJob
from sagemaker.core.shapes import (
    ContainerDefinition,
    ProductionVariant,
    TransformInput,
    TransformDataSource,
    TransformS3DataSource,
    TransformOutput,
    TransformResources,
)
from sagemaker.core.common_utils import name_from_base

model_name = name_from_base("scikit-iris-model")

# NOTE: Marketplace model packages require network isolation. The v2
# high-level `ModelPackage.deploy()` / `.transformer()` set this implicitly;
# with the sagemaker-core `Model` we must pass `enable_network_isolation=True`
# explicitly, otherwise CreateModel fails with "EnableNetworkIsolation must be
# set to true for using a product from AWS Marketplace."
model = Model.create(
    model_name=model_name,
    primary_container=ContainerDefinition(
        model_package_name=modelpackage_arn,
    ),
    execution_role_arn=role,
    enable_network_isolation=True,
    region=region,
)
print("Created model " + model_name)

sagemaker.config INFO - Not applying SDK defaults from location: /Library/Application Support/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /Users/lucasjia/Library/Application Support/sagemaker/config.yaml


                    INFO     Creating model resource.                                            ]8;id=4179111;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4179112;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20593\20593]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-1                                  ]8;id=4179119;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=4179120;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/utils/utils.py#375\375]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=4179125;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=4179126;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

                    INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=4179131;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=4179132;file:///Users/lucasjia/pysdk/pysdk-workspace/.v3test-venv/lib/python3.12/site-packages/botocore/credentials.py#1392\1392]8;;\

Created model scikit-iris-model-2026-07-16-19-57-17-470


## Batch Transform Job

Now let's use the model built to run a batch inference job and verify it works.

### Batch Transform Input Preparation

The snippet below is removing the "label" column (column indexed at 0) and retaining the rest to be batch transform's input. 

NOTE: This is the same training data, which is a no-no from a statistical/ML science perspective. But the aim of this notebook is to demonstrate how things work end-to-end.

In [5]:
import pandas as pd

## Remove first column that contains the label
shape = pd.read_csv("data/training/iris.csv", header=None).drop([0], axis=1)

TRANSFORM_WORKDIR = "data/transform"
shape.to_csv(TRANSFORM_WORKDIR + "/batchtransform_test.csv", index=False, header=False)

transform_input = (
    sagemaker_session.upload_data(TRANSFORM_WORKDIR, key_prefix=batch_inference_input_prefix)
    + "/batchtransform_test.csv"
)
print("Transform input uploaded to " + transform_input)

Transform input uploaded to s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/batch-inference-input-data/batchtransform_test.csv


In [6]:
# v3: `model.transformer(1, "ml.m4.xlarge")` + `transformer.transform(...)` +
# `transformer.wait()` -> `TransformJob.create(...)` + `transform_job.wait()`.
# Reuses the same core Model created above. Unlike the v2 transformer (whose
# `output_path` is auto-generated), the v3 TransformJob requires an explicit
# `transform_output.s3_output_path`, so we set one and reuse it below when
# inspecting the output.
transform_job_name = name_from_base("scikit-iris-transform")

default_bucket = sagemaker_session.default_bucket()
default_bucket_prefix = sagemaker_session.default_bucket_prefix
transform_output_key = "{}/batch-transform-output".format(common_prefix)
if default_bucket_prefix:
    transform_output_key = f"{default_bucket_prefix}/{transform_output_key}"
transform_output_path = f"s3://{default_bucket}/{transform_output_key}"

transform_job = TransformJob.create(
    transform_job_name=transform_job_name,
    model_name=model_name,
    transform_input=TransformInput(
        data_source=TransformDataSource(
            s3_data_source=TransformS3DataSource(
                s3_data_type="S3Prefix",
                s3_uri=transform_input,
            )
        ),
        content_type="text/csv",
    ),
    transform_output=TransformOutput(
        s3_output_path=transform_output_path,
    ),
    transform_resources=TransformResources(
        instance_type="ml.m4.xlarge",
        instance_count=1,
    ),
    region=region,
)
transform_job.wait()

print("Batch Transform output saved to " + transform_output_path)

[07/16/26 12:57:18] INFO     Creating transform_job resource.                                    ]8;id=4179138;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4179139;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#32261\32261]8;;\

Output()

[07/16/26 13:01:36] INFO     Final Resource Status: Completed                                    ]8;id=4179145;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4179146;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#32483\32483]8;;\

Batch Transform output saved to s3://sagemaker-us-west-1-729646638167/DEMO-scikit-byo-iris/batch-transform-output


#### Inspect the Batch Transform Output in S3

In [7]:
# v3: the v2 code derived the output location from `transformer.output_path`.
# In v3 we explicitly set `transform_output_path` above, so we read the
# `<input-file>.out` object from that prefix.
file_key = "{}/{}.out".format(transform_output_key, "batchtransform_test.csv")

s3_client = sagemaker_session.boto_session.client("s3")

response = s3_client.get_object(Bucket=default_bucket, Key=file_key)
response_bytes = response["Body"].read().decode("utf-8")
print(response_bytes)

setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica


## Live Inference Endpoint

Now we demonstrate the creation of an endpoint for live inference

In [8]:
# v3: `model.deploy(1, "ml.m4.xlarge", endpoint_name=...)` ->
# `EndpointConfig.create(...ProductionVariant...)` + `Endpoint.create(...)` +
# `endpoint.wait_for_status("InService")`.
endpoint_name = name_from_base("scikit-model")

EndpointConfig.create(
    endpoint_config_name=endpoint_name,
    production_variants=[
        ProductionVariant(
            variant_name="AllTraffic",
            model_name=model_name,
            initial_instance_count=1,
            instance_type="ml.m4.xlarge",
            initial_variant_weight=1.0,
        )
    ],
    region=region,
)

endpoint = Endpoint.create(
    endpoint_name=endpoint_name,
    endpoint_config_name=endpoint_name,
    region=region,
)
endpoint.wait_for_status("InService")
print("Endpoint in service: " + endpoint.endpoint_name)

                    INFO     Creating endpoint_config resource.                                  ]8;id=4185190;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4185191;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11069\11069]8;;\

                    INFO     Creating endpoint resource.                                         ]8;id=4185197;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4185198;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10228\10228]8;;\

Output()

[07/16/26 13:03:30] INFO     Final Resource Status: InService                                    ]8;id=4185204;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4185205;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10484\10484]8;;\

Endpoint in service: scikit-model-2026-07-16-20-01-37-550


### Choose some data and use it for a prediction

In order to do some predictions, we'll extract some of the data we used for training and do predictions against it. This is, of course, bad statistical practice, but a good way to see how the mechanism works.


In [9]:
TRAINING_WORKDIR = "data/training"
shape = pd.read_csv(TRAINING_WORKDIR + "/iris.csv", header=None)

import itertools

a = [50 * i for i in range(3)]
b = [40 + i for i in range(10)]
indices = [i + j for i, j in itertools.product(a, b)]

test_data = shape.iloc[indices[:-1]]
test_X = test_data.iloc[:, 1:]
test_y = test_data.iloc[:, 0]

In [10]:
# v3: `Predictor(endpoint_name, serializer=CSVSerializer()).predict(test_X.values)`
# -> `Endpoint.invoke(body=<csv bytes>, content_type="text/csv")`. We serialize the
# numpy rows to CSV ourselves (what CSVSerializer did under the hood in v2).
import io
import numpy as np


def _to_csv_bytes(array):
    buffer = io.StringIO()
    np.savetxt(buffer, np.array(array), delimiter=",", fmt="%s")
    return buffer.getvalue().encode("utf-8")


csv_input = _to_csv_bytes(test_X.values)

response = endpoint.invoke(
    body=csv_input,
    content_type="text/csv",
    accept="text/csv",
)
# v3 `Endpoint.invoke` returns an `InvokeEndpointOutput`; `.body` is a boto3
# StreamingBody (no deserializer configured), so read + decode it directly.
result_body = response.body
if hasattr(result_body, "read"):
    result_body = result_body.read()
if isinstance(result_body, bytes):
    result_body = result_body.decode("utf-8")
print(result_body)

setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
setosa
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
versicolor
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica
virginica



### Cleanup endpoint


In [11]:
# v3: `sagemaker_session.delete_endpoint(...)` / `delete_endpoint_config(...)` /
# `model.delete_model()` -> core resource `.delete()` calls.
endpoint.delete()
EndpointConfig.get(endpoint_name, region=region).delete()
model.delete()

[07/16/26 13:03:31] INFO     Deleting Endpoint - scikit-model-2026-07-16-20-01-37-550            ]8;id=4187833;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4187834;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#10428\10428]8;;\

                    INFO     Deleting EndpointConfig - scikit-model-2026-07-16-20-01-37-550      ]8;id=4187840;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4187841;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#11220\11220]8;;\

                    INFO     Deleting Model - scikit-iris-model-2026-07-16-19-57-17-470          ]8;id=4187847;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py\resources.py]8;;\:]8;id=4187848;file:///Users/lucasjia/pysdk/pysdk-workspace/sagemaker-python-sdk-lucas/sagemaker-core/src/sagemaker/core/resources.py#20740\20740]8;;\

## Notebook CI Test Results

This notebook was tested in multiple regions. The test results are as follows, except for us-west-2 which is shown at the top of the notebook.

![This us-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This us-east-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-east-2/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This us-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/us-west-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ca-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ca-central-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This sa-east-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/sa-east-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This eu-west-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This eu-west-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-2/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This eu-west-3 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-west-3/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This eu-central-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-central-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This eu-north-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/eu-north-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ap-southeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ap-southeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-southeast-2/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ap-northeast-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ap-northeast-2 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-northeast-2/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)

![This ap-south-1 badge failed to load. Check your device's internet connectivity, otherwise the service is currently unavailable](https://prod.us-west-2.tcx-beacon.docs.aws.dev/sagemaker-nb/ap-south-1/deploy_and_monitor|sm-marketplace_using_model_package_arn|sm-marketplace_using_model_package_arn.ipynb)
